In [1]:
import pandas as pd
import numpy as np

# Load clean dataset

In [2]:
data = pd.read_csv('Cleaned_data_Telco_Customer_Churn.csv')

In [3]:
data.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# Split Features and Target Variable

In [4]:
x=data.drop(columns = 'Churn')
y = data['Churn'].map({'Yes': 1, 'No': 0})

In [5]:
from sklearn.model_selection import train_test_split,GridSearchCV
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [6]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# Define Categorical Columns

In [7]:
yes_no_cols_list = ['Partner','Dependents','PhoneService','MultipleLines','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','PaperlessBilling']

In [8]:
strict_categories = [['No', 'Yes'] for _ in yes_no_cols_list]

In [9]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',OrdinalEncoder(categories=strict_categories),yes_no_cols_list),
    ('tnf2',OrdinalEncoder(categories =[['Month-to-month','One year','Two year']]),['Contract']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','InternetService','PaymentMethod',]),
    ('tnf4',StandardScaler(),['tenure', 'MonthlyCharges', 'TotalCharges'])
])

# Logistic Regression Model with Pipeline

In [10]:
clf = LogisticRegression(max_iter=1000)

In [11]:
lr_pipe = make_pipeline(transformer,clf)

# Define Hyperparameters for Logistic Regression

## Apply GridSearchCV on Logistic Regression

In [12]:
lr_params = {
    'logisticregression__C':[0.01,0.1,1,10]
}

In [13]:
grid_lr = GridSearchCV(
    estimator=lr_pipe,
    param_grid=lr_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [14]:
grid_lr.fit(x_train, y_train)

,estimator,Pipeline(step..._iter=1000))])
,param_grid,"{'logisticregression__C': [0.01, 0.1, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('tnf1', ...), ('tnf2', ...), ...]"


# Predict Using Logistic Regression Model and Evaluate

In [15]:
lr_pred = grid_lr.predict(x_test)

In [16]:
lr_accuracy = accuracy_score(y_test, lr_pred)

print("Logistic Regression Best Params:", grid_lr.best_params_)
print("Logistic Regression Accuracy:", lr_accuracy)
print(classification_report(y_test, lr_pred))

Logistic Regression Best Params: {'logisticregression__C': 1}
Logistic Regression Accuracy: 0.8176011355571328
              precision    recall  f1-score   support

           0       0.86      0.89      0.88      1036
           1       0.67      0.60      0.64       373

    accuracy                           0.82      1409
   macro avg       0.77      0.75      0.76      1409
weighted avg       0.81      0.82      0.81      1409



# Random Forest Model with Pipeline

In [17]:
rf_clf = RandomForestClassifier(random_state=42)

In [18]:
rf_pipe = make_pipeline(transformer, rf_clf)

In [19]:
rf_params = {
    'randomforestclassifier__n_estimators': [100, 200],
    'randomforestclassifier__max_depth': [None, 5, 10],
    'randomforestclassifier__min_samples_split': [2, 5],
    'randomforestclassifier__min_samples_leaf': [1, 2]
}

In [20]:
grid_rf = GridSearchCV(
    estimator=rf_pipe,
    param_grid=rf_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Predict Using Random Forest Model and Evaluate

In [ ]:
grid_rf.fit(x_train, y_train)

In [ ]:
rf_pred = grid_rf.predict(x_test)

In [ ]:
rf_accuracy = accuracy_score(y_test, rf_pred)

print("Random Forest Best Params:", grid_rf.best_params_)
print("Random Forest Accuracy:", rf_accuracy)
print(classification_report(y_test, rf_pred))

# Compare Logistic Regression and Random Forest

In [ ]:
print("Logistic Regression Accuracy:", lr_accuracy)
print("Random Forest Accuracy:", rf_accuracy)

if rf_accuracy > lr_accuracy:
    best_model = grid_rf.best_estimator_
    print("Best Model: Random Forest")
else:
    best_model = grid_lr.best_estimator_
    print("Best Model: Logistic Regression")

# Export Final Best Pipeline Using Joblib

In [ ]:
joblib.dump(best_model, 'churn_pipeline.pkl')

# testing

In [ ]:
loaded_model = joblib.load('churn_pipeline.pkl')

sample_prediction = loaded_model.predict(x_test.head())

sample_prediction